In [1]:
import uproot
import awkward as ak
import dask_awkward as dak
import dask
import numpy as np
import pandas as pd  # Import que estava faltando
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
import mplhep as hep
import vector

vector.register_awkward() 
file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoKstar2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoKstarMuMu_NoFilter_2022/260427_132612/0000/*.root"

print("Lendo arquivos e montando o grafo Lazy...")
df = uproot.dask(
    {file_path: "Events"},
    library="ak"
) 

# ==============================================================================
# 1. TRUTH MATCHING (INTACTA)
# ==============================================================================
def get_reco_truth_mask(df):
    # --- 1. Constantes PDG ---
    PDG_B0        = 511
    PDG_KSTAR     = 313
    PDG_KAON_plus = 321
    PDG_PION_neg  = -211
    PDG_MUON_plus = -13
    PDG_MUON_neg  = 13
    PDG_PHOTON    = 22

    # Pega os índices genéticos dos candidatos RECO (Shape: BToTrkTrkMuMu)
    mu1_idx  = df["BPHMuon_genPartIdx"][df["BToTrkTrkMuMu_l1_idx"]]
    mu2_idx  = df["BPHMuon_genPartIdx"][df["BToTrkTrkMuMu_l2_idx"]]
    trk1_idx = df["Track_genPartIdx"][df["BToTrkTrkMuMu_trk1_idx"]]
    trk2_idx = df["Track_genPartIdx"][df["BToTrkTrkMuMu_trk2_idx"]]

    pdg_id     = df["BPHGenPart_pdgId"]
    mother_idx = df["BPHGenPart_genPartIdxMother"]

    # Mascara índices inválidos (-1) para evitar IndexError.
    # O -1 vira 'None' e a operação propaga o 'None' de forma segura.
    mu1_idx_safe  = dak.mask(mu1_idx, mu1_idx >= 0)
    mu2_idx_safe  = dak.mask(mu2_idx, mu2_idx >= 0)
    trk1_idx_safe = dak.mask(trk1_idx, trk1_idx >= 0)
    trk2_idx_safe = dak.mask(trk2_idx, trk2_idx >= 0)

    # Verificação de Identidade (PDG ID)
    mu1_pdg  = pdg_id[mu1_idx_safe]
    mu2_pdg  = pdg_id[mu2_idx_safe]
    trk1_pdg = pdg_id[trk1_idx_safe]
    trk2_pdg = pdg_id[trk2_idx_safe]

    flavor_match = (trk1_pdg == PDG_KAON_plus) & (trk2_pdg == PDG_PION_neg) & \
                   (mu1_pdg == PDG_MUON_plus) & (mu2_pdg == PDG_MUON_neg)

    # Linhagem do Hadron (K, Pi -> K* -> B0)
    mom_trk1_idx = mother_idx[trk1_idx_safe]
    mom_trk2_idx = mother_idx[trk2_idx_safe]
    
    mom_trk1_idx_safe = dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)
    mom_trk2_idx_safe = dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)
    
    mom_trk1_pdg = pdg_id[mom_trk1_idx_safe]
    mom_trk2_pdg = pdg_id[mom_trk2_idx_safe]
    
    # K e Pi devem vir do mesmo objeto K*0
    match_kstar = (mom_trk1_idx == mom_trk2_idx) & (mom_trk1_idx >= 0) & \
                  (mom_trk1_pdg == PDG_KSTAR) & (mom_trk2_pdg == PDG_KSTAR)
    
    # O K*0 deve vir de um B0
    gmom_trk1_idx = mother_idx[mom_trk1_idx_safe]
    gmom_trk2_idx = mother_idx[mom_trk2_idx_safe]
    
    gmom_trk1_idx_safe = dak.mask(gmom_trk1_idx, gmom_trk1_idx >= 0)
    gmom_trk2_idx_safe = dak.mask(gmom_trk2_idx, gmom_trk2_idx >= 0)
    
    gmom_trk1_pdg = pdg_id[gmom_trk1_idx_safe]
    gmom_trk2_pdg = pdg_id[gmom_trk2_idx_safe]
    
    match_kstar_to_b0 = (gmom_trk1_idx == gmom_trk2_idx) & (gmom_trk1_idx >= 0) & \
                        (gmom_trk1_pdg == PDG_B0) & (gmom_trk2_pdg == PDG_B0)

    # Linhagem dos Múons
    mom_mu1_idx = mother_idx[mu1_idx_safe]
    mom_mu2_idx = mother_idx[mu2_idx_safe]
    
    mom_mu1_idx_safe = dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)
    mom_mu2_idx_safe = dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)
    
    mom_mu1_pdg = pdg_id[mom_mu1_idx_safe]
    mom_mu2_pdg = pdg_id[mom_mu2_idx_safe]
    
    gmom_mu1_idx = mother_idx[mom_mu1_idx_safe]
    gmom_mu2_idx = mother_idx[mom_mu2_idx_safe]

    # Muon: Mãe é o B0 do hadron OU (Mãe é fóton e Avó é o B0 do hadron)
    mu1_from_same_b0 = (mom_mu1_idx == gmom_trk1_idx) | \
                       ((mom_mu1_pdg == PDG_PHOTON) & (gmom_mu1_idx == gmom_trk1_idx))

    mu2_from_same_b0 = (mom_mu2_idx == gmom_trk1_idx) | \
                       ((mom_mu2_pdg == PDG_PHOTON) & (gmom_mu2_idx == gmom_trk1_idx))

    # --- Máscara Final ---
    full_mask = flavor_match & match_kstar & match_kstar_to_b0 & \
                mu1_from_same_b0 & mu2_from_same_b0
    
    # Preenche tudo que virou 'None' com False
    return dak.fill_none(full_mask, False)


# ==============================================================================
# 2. APLICAÇÃO DE CORTES E FILTROS
# ==============================================================================
def apply_selection(df, mode="mc"):
    kstar_pdg = 0.892
    mask_kpi_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) <= 0.150
    mask_pik_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg) <= 0.150
    is_kpi_closer = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) < \
                    abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg)
    
    # Aqui chamamos o truth matching que você definiu para usar junto aos cortes!
    truth_mask = get_reco_truth_mask(df)
    
    mask = (
        (mask_kpi_window | mask_pik_window) &         
        (is_kpi_closer) &
        (df['BPHMuon_pt'][df["BToTrkTrkMuMu_l1_idx"]] > 2.0) & 
        (df['BPHMuon_pt'][df["BToTrkTrkMuMu_l2_idx"]] > 2.0) & 
        (abs(df['BPHMuon_eta'][df["BToTrkTrkMuMu_l1_idx"]]) < 2.4) & 
        (abs(df['BPHMuon_eta'][df["BToTrkTrkMuMu_l2_idx"]]) < 2.4) &
        (df['BToTrkTrkMuMu_fit_trk1_pt'] > 1.5) &
        (df['BToTrkTrkMuMu_fit_trk2_pt'] > 1.5) &
        truth_mask # Inserido na lógica de filtragem
    )

    cols_to_keep = [
        'BToTrkTrkMuMu_fit_mass_Kpi',
        'BToTrkTrkMuMu_mll_fullfit',
    ]
        
    return df[cols_to_keep][mask]

# ==============================================================================
# 3. COMPUTAÇÃO DASK -> AWKWARD -> PANDAS
# ==============================================================================
# O fluxo agora é linear e claro.
df_selected = apply_selection(df)

print("Iniciando a computação do Dask Awkward para a memória...")
with ProgressBar():
    awkward_array = df_selected.compute()

# Converte o array Awkward filtrado (que pode ser list-like/jagged) em DataFrame do Pandas flat.
df_mc = ak.to_dataframe(awkward_array).reset_index(drop=True)

# ==============================================================================
# 4. ANÁLISE DE BINS NO PANDAS
# ==============================================================================
df_mc['q2'] = df_mc['BToTrkTrkMuMu_mll_fullfit']**2

bins = [1.1, 2, 4.3, 6, 8.68, 10.09, 12.86, 14.18, 16, 19]
labels = [0, 1, 2, 3, 4, 5, 6, 7, 8]

df_mc['bin_index'] = pd.cut(df_mc['q2'], bins=bins, labels=labels, right=False).astype(float)

print(f"\nTotal de partículas (candidatos) no DF RECO: {len(df_mc)}")

for i in range(9):  
    is_bin = df_mc["bin_index"] == i    
    df_filtrado = df_mc[is_bin]    
    print(f"Bin {i}: {len(df_filtrado)} candidatos B0 encontrados")

Lendo arquivos e montando o grafo Lazy...
Iniciando a computação do Dask Awkward para a memória...
[########################################] | 100% Completed | 205.86 ms
[########################################] | 100% Completed | 20.62 s

Total de partículas (candidatos) no DF RECO: 5961
Bin 0: 147 candidatos B0 encontrados
Bin 1: 435 candidatos B0 encontrados
Bin 2: 364 candidatos B0 encontrados
Bin 3: 752 candidatos B0 encontrados
Bin 4: 557 candidatos B0 encontrados
Bin 5: 1213 candidatos B0 encontrados
Bin 6: 576 candidatos B0 encontrados
Bin 7: 760 candidatos B0 encontrados
Bin 8: 663 candidatos B0 encontrados


In [3]:
import uproot
import awkward as ak
import dask_awkward as dak
import dask
import numpy as np
import pandas as pd  # Import que estava faltando
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
import mplhep as hep
import vector

vector.register_awkward() 
file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoJpsiKstar_Jpsito2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoJpsiKstar_NoFilter_2022_retry1/260427_173326/0000/*.root"

print("Lendo arquivos e montando o grafo Lazy...")
df = uproot.dask(
    {file_path: "Events"},
    library="ak"
) 

# ==============================================================================
# 1. TRUTH MATCHING (INTACTA)
# ==============================================================================
def get_reco_truth_mask(events):
    # Constantes PDG
    PDG_B0        =  511
    PDG_KSTAR     =  313
    PDG_KAON_plus =  321
    PDG_PION_neg  = -211
    PDG_PHOTON    =  22
    PDG_JPSI      =  443
    PDG_MUON_plus = -13
    PDG_MUON_neg  =  13

    # 1. Carregar as arrays alvo (Sem usar .array() para manter a avaliação lazy do Dask)
    pdg_ids = events["BPHGenPart_pdgId"]
    mom_idx = events["BPHGenPart_genPartIdxMother"]

    muon_genPartIdx = events["BPHMuon_genPartIdx"]
    track_genPartIdx = events["Track_genPartIdx"]

    # 2. Carregar as arrays de índices
    l1_idx_arr   = events["BToTrkTrkMuMu_l1_idx"]
    l2_idx_arr   = events["BToTrkTrkMuMu_l2_idx"]
    trk1_idx_arr = events["BToTrkTrkMuMu_trk1_idx"]
    trk2_idx_arr = events["BToTrkTrkMuMu_trk2_idx"]

    # 3. Slicing lazy das arrays
    mu1_idx  = muon_genPartIdx[l1_idx_arr]
    mu2_idx  = muon_genPartIdx[l2_idx_arr]
    trk1_idx = track_genPartIdx[trk1_idx_arr]
    trk2_idx = track_genPartIdx[trk2_idx_arr]

    # Usando dak.mask no lugar de ak.mask
    mu1_pdg  = pdg_ids[dak.mask(mu1_idx, mu1_idx >= 0)]
    mu2_pdg  = pdg_ids[dak.mask(mu2_idx, mu2_idx >= 0)]
    trk1_pdg = pdg_ids[dak.mask(trk1_idx, trk1_idx >= 0)]
    trk2_pdg = pdg_ids[dak.mask(trk2_idx, trk2_idx >= 0)]

    flavor_match = (trk1_pdg == PDG_KAON_plus) & (trk2_pdg == PDG_PION_neg) & \
                   (mu1_pdg == PDG_MUON_plus) & (mu2_pdg == PDG_MUON_neg)

    mom_trk1_idx = mom_idx[dak.mask(trk1_idx, trk1_idx >= 0)]
    mom_trk2_idx = mom_idx[dak.mask(trk2_idx, trk2_idx >= 0)]

    mom_trk1_pdg = pdg_ids[dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)]
    mom_trk2_pdg = pdg_ids[dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)]

    match_kstar = (mom_trk1_idx == mom_trk2_idx) & (mom_trk1_idx >= 0) & (mom_trk2_idx >= 0) & \
                  (mom_trk1_pdg == PDG_KSTAR) & (mom_trk2_pdg == PDG_KSTAR)

    gmom_trk1_idx = mom_idx[dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)]
    gmom_trk2_idx = mom_idx[dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)]

    gmom_trk1_pdg = pdg_ids[dak.mask(gmom_trk1_idx, gmom_trk1_idx >= 0)]
    gmom_trk2_pdg = pdg_ids[dak.mask(gmom_trk2_idx, gmom_trk2_idx >= 0)]

    match_kstar_to_b0 = (gmom_trk1_idx == gmom_trk2_idx) & (gmom_trk1_idx >= 0) & (gmom_trk2_idx >= 0) & \
                        (gmom_trk1_pdg == PDG_B0) & (gmom_trk2_pdg == PDG_B0)

    mom_mu1_idx = mom_idx[dak.mask(mu1_idx, mu1_idx >= 0)]
    mom_mu2_idx = mom_idx[dak.mask(mu2_idx, mu2_idx >= 0)]

    gmom_mu1_idx = mom_idx[dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)]
    gmom_mu2_idx = mom_idx[dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)]

    mom_mu1_pdg = pdg_ids[dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)]
    mom_mu2_pdg = pdg_ids[dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)]

    gmom_mu1_pdg = pdg_ids[dak.mask(gmom_mu1_idx, gmom_mu1_idx >= 0)]
    gmom_mu2_pdg = pdg_ids[dak.mask(gmom_mu2_idx, gmom_mu2_idx >= 0)]

    ggmom_mu1_idx = mom_idx[dak.mask(gmom_mu1_idx, gmom_mu1_idx >= 0)]
    ggmom_mu2_idx = mom_idx[dak.mask(gmom_mu2_idx, gmom_mu2_idx >= 0)]

    ggmom_mu1_pdg = pdg_ids[dak.mask(ggmom_mu1_idx, ggmom_mu1_idx >= 0)]
    ggmom_mu2_pdg = pdg_ids[dak.mask(ggmom_mu2_idx, ggmom_mu2_idx >= 0)]

    mu1_match = (mom_mu1_idx == gmom_trk1_idx) | \
                ((mom_mu1_pdg == PDG_PHOTON) & (gmom_mu2_pdg == PDG_JPSI) & (ggmom_mu1_idx == gmom_trk1_idx))

    mu2_match = (mom_mu2_idx == gmom_trk1_idx) | \
                ((mom_mu2_pdg == PDG_PHOTON) & (gmom_mu2_pdg == PDG_JPSI) & (ggmom_mu2_idx == gmom_trk1_idx))

    truth_match_final = flavor_match & match_kstar & match_kstar_to_b0 & mu1_match & mu2_match

    # Usando dak.fill_none para substituir os valores 'None' gerados pelas masks por 'False'
    return dak.fill_none(truth_match_final, False)


# ==============================================================================
# 2. APLICAÇÃO DE CORTES E FILTROS
# ==============================================================================
def apply_selection(df, mode="mc"):
    kstar_pdg = 0.892
    mask_kpi_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) <= 0.150
    mask_pik_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg) <= 0.150
    is_kpi_closer = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) < \
                    abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg)
    
    # Aqui chamamos o truth matching que você definiu para usar junto aos cortes!
    truth_mask = get_reco_truth_mask(df)
    
    mask = (
        (mask_kpi_window | mask_pik_window) &         
        (is_kpi_closer) &
        (df['BPHMuon_pt'][df["BToTrkTrkMuMu_l1_idx"]] > 2.0) & 
        (df['BPHMuon_pt'][df["BToTrkTrkMuMu_l2_idx"]] > 2.0) & 
        (abs(df['BPHMuon_eta'][df["BToTrkTrkMuMu_l1_idx"]]) < 2.4) & 
        (abs(df['BPHMuon_eta'][df["BToTrkTrkMuMu_l2_idx"]]) < 2.4) &
        (df['BToTrkTrkMuMu_fit_trk1_pt'] > 1.5) &
        (df['BToTrkTrkMuMu_fit_trk2_pt'] > 1.5) &
        truth_mask # Inserido na lógica de filtragem
    )

    cols_to_keep = [
        'BToTrkTrkMuMu_fit_mass_Kpi',
        'BToTrkTrkMuMu_mll_fullfit',
    ]
        
    return df[cols_to_keep][mask]

# ==============================================================================
# 3. COMPUTAÇÃO DASK -> AWKWARD -> PANDAS
# ==============================================================================
# O fluxo agora é linear e claro.
df_selected = apply_selection(df)

print("Iniciando a computação do Dask Awkward para a memória...")
with ProgressBar():
    awkward_array = df_selected.compute()

# Converte o array Awkward filtrado (que pode ser list-like/jagged) em DataFrame do Pandas flat.
df_mc = ak.to_dataframe(awkward_array).reset_index(drop=True)

# ==============================================================================
# 4. ANÁLISE DE BINS NO PANDAS
# ==============================================================================
df_mc['q2'] = df_mc['BToTrkTrkMuMu_mll_fullfit']**2

bins = [1.1, 2, 4.3, 6, 8.68, 10.09, 12.86, 14.18, 16, 19]
labels = [0, 1, 2, 3, 4, 5, 6, 7, 8]

df_mc['bin_index'] = pd.cut(df_mc['q2'], bins=bins, labels=labels, right=False).astype(float)

print(f"\nTotal de partículas Controle no DF RECO: {len(df_mc)}")

for i in range(9):  
    is_bin = df_mc["bin_index"] == i    
    df_filtrado = df_mc[is_bin]    
    print(f"Bin {i}: {len(df_filtrado)} candidatos B0 encontrados")

Lendo arquivos e montando o grafo Lazy...
Iniciando a computação do Dask Awkward para a memória...
[########################################] | 100% Completed | 206.42 ms
[########################################] | 100% Completed | 71.84 s

Total de partículas Controle no DF RECO: 6362
Bin 0: 0 candidatos B0 encontrados
Bin 1: 1 candidatos B0 encontrados
Bin 2: 6 candidatos B0 encontrados
Bin 3: 158 candidatos B0 encontrados
Bin 4: 6127 candidatos B0 encontrados
Bin 5: 70 candidatos B0 encontrados
Bin 6: 0 candidatos B0 encontrados
Bin 7: 0 candidatos B0 encontrados
Bin 8: 0 candidatos B0 encontrados
